# Legal Notice Multi-Class Document Classifier
### Programming For AI -- End-Term Lab Assessment | Spring 2026
**Riphah International University, Lahore Campus**

---
**Dataset:** `legal_notices.csv` -- 600 samples, 3 classes  
**Models:** Logistic Regression vs Multinomial Naive Bayes  
**Features:** Bag-of-Words vs TF-IDF


In [ ]:
# CELL 1 -- Imports
import json, os, sys, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, classification_report, confusion_matrix)

# Add src/ to Python path so we can import our custom modules
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
from preprocess import preprocess_series
from features import (build_bow_vectorizer, build_tfidf_vectorizer,
                       fit_transform_vectorizer, print_top_terms)
from evaluate import train_and_time, predict_and_time, compute_metrics, plot_confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid')
print('All imports successful.')


In [ ]:
# CELL 2 -- Load Configuration
# All hyperparameters are read from config.json; nothing hardcoded in the notebook.
with open('config.json', 'r') as f:
    config = json.load(f)

SEED      = config['random_seed']
TEST_SIZE = config['test_size']
MAX_FEAT  = config['max_features']
LR_C      = config['model_1']['C']
LR_ITER   = config['model_1']['max_iter']
NB_ALPHA  = config['model_2']['alpha']

print('Configuration loaded:')
print(json.dumps(config, indent=2))


---
## Part 1 -- Problem Analysis and Design [15 marks]

### Task 1.1 -- Problem Framing

**ML Task Type:** Multi-class Text Classification -- supervised, discrete output (3 labels), so regression and clustering are inappropriate.

**Assumptions:**
1. Category labels in the CSV are correct and complete (no mislabelling).
2. Word-level features (BoW/TF-IDF) are sufficient; semantic embeddings are not required.
3. The 200-per-class balance reflects production distribution.
4. Notice length is consistent enough that preprocessing does not collapse any sample to empty.

**Constraints:**
1. **Interpretability:** Legal startup needs explainable decisions. Logistic Regression coefficients map to feature importance.
2. **Inference Speed:** Notices must be classified in real time (<100ms). Both LR and NB are sub-millisecond.
3. **Data Size:** 600 samples -- deep learning would overfit; classical ML is more robust here.

**Primary Evaluation Metric:** **Macro F1-Score** -- equally weights all 3 classes. In legal contexts, missing a category (low recall) is more costly than a false alarm, so F1 balances precision and recall better than accuracy alone.

---
### Task 1.2 -- Approach Comparison

| Criterion | Logistic Regression + TF-IDF | Multinomial Naive Bayes + BoW |
|---|---|---|
| **Description** | Linear classifier; optimises cross-entropy with L2 regularisation | Probabilistic generative model; applies Bayes theorem with word counts |
| **Strengths** | Handles correlated features; TF-IDF down-weights noise | Extremely fast; works well on small datasets |
| **Weaknesses** | Slower than NB; needs feature tuning | Assumes feature independence (violated in legal text) |
| **Best For** | Legal text with co-occurring domain terms | Simple fast baselines |

**Final Selection:** LR + TF-IDF is expected to outperform because legal notices contain highly correlated vocabulary (e.g., 'patent', 'infringe', 'cease' co-occur in Class B). LR handles correlation; NB assumes independence which is violated here. Both are implemented and compared empirically below.


---
## Part 2 -- Data Handling and Preprocessing [20 marks]

### Task 2.1 -- Exploratory Data Analysis (EDA)

In [ ]:
# CELL 3 -- Load Dataset
df = pd.read_csv('data/raw/legal_notices.csv')

print('='*60)
print('DATASET OVERVIEW')
print('='*60)
print(f'Shape         : {df.shape}')
print(f'Total Samples : {len(df)}')
print(f'Columns       : {list(df.columns)}')
print()
print('First 10 rows:')
df.head(10)


In [ ]:
# CELL 4 -- EDA Statistics
# Class distribution -- checks for class imbalance
class_counts = df['category'].value_counts().sort_index()
print('CLASS DISTRIBUTION:')
print(class_counts)

# Null values -- missing data can crash preprocessing
print('\nNULL VALUES:')
print(df.isnull().sum())

# Notice length analysis -- helps identify outliers
df['word_count']  = df['notice'].str.split().str.len()
df['char_length'] = df['notice'].str.len()
print('\nNOTICE LENGTH (words) per class:')
print(df.groupby('category')['word_count'].describe().round(2))
print(f'\nOverall average length : {df["word_count"].mean():.1f} words')

# Data quality check
print('\nDATA QUALITY OBSERVATIONS:')
print(f'  Duplicate notices : {df["notice"].duplicated().sum()}')
print(f'  Empty notices     : {(df["notice"].str.strip() == "").sum()}')
print('  WARNING: Some Class C notices cite statutes mismatched to their domain')
print('  (e.g. Clean Air Act cited in a telecom context). This suggests synthetic')
print('  templated data, which may cause the model to learn spurious patterns.')


In [ ]:
# CELL 5 -- EDA Visualisations
os.makedirs('results', exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: Class distribution bar chart
labels = ['A: Contract\nDispute', 'B: IP\nClaim', 'C: Regulatory\nCompliance']
counts = [class_counts['A'], class_counts['B'], class_counts['C']]
colors = ['#4C72B0', '#DD8452', '#55A868']
bars = axes[0].bar(labels, counts, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Category', fontsize=11)
axes[0].set_ylabel('Number of Samples', fontsize=11)
axes[0].set_ylim(0, 250)
for bar, cnt in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(cnt), ha='center', va='bottom', fontweight='bold', fontsize=12)

# Plot 2: Word count histogram per class
for cat, color in zip(['A','B','C'], colors):
    axes[1].hist(df[df['category']==cat]['word_count'],
                 bins=25, alpha=0.6, label=f'Class {cat}', color=color, edgecolor='white')
axes[1].set_title('Notice Length Distribution (Word Count)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Word Count', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].legend(title='Category')

plt.suptitle('Legal Notices -- EDA', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plots saved to results/eda_plots.png')


### Task 2.2 -- Text Preprocessing Pipeline

In [ ]:
# CELL 6 -- Preprocessing
# preprocess_series() is in src/preprocess.py
# Steps: HTML removal -> lowercase -> punctuation removal -> tokenise
#        -> stopword removal (NLTK) -> lemmatisation
# Lemmatisation chosen over stemming to preserve real legal base forms

print('Applying preprocessing pipeline...')
df['clean_notice'] = preprocess_series(df['notice'])

# Save processed data for reproducibility
df.to_csv('data/processed/legal_notices_clean.csv', index=False)

print('\nBEFORE preprocessing:')
print(df['notice'].iloc[0][:250])
print('\nAFTER preprocessing:')
print(df['clean_notice'].iloc[0][:250])

df['clean_word_count'] = df['clean_notice'].str.split().str.len()
print(f'\nAvg words BEFORE: {df["word_count"].mean():.1f}')
print(f'Avg words AFTER : {df["clean_word_count"].mean():.1f}')
print('(Reduction expected from stopword removal and punctuation stripping)')


### Task 2.3 -- Feature Extraction

In [ ]:
# CELL 7 -- Train/Test Split (stratified, seed from config.json)
X = df['clean_notice']
y = df['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,    # from config.json
    random_state=SEED,      # from config.json -- NOT hardcoded
    stratify=y              # ensures each class equally represented in both splits
)

print(f'Train size : {len(X_train)} | Test size: {len(X_test)}')
print(f'Train dist : {y_train.value_counts().to_dict()}')
print(f'Test  dist : {y_test.value_counts().to_dict()}')


In [ ]:
# CELL 8 -- Feature Extraction: BoW and TF-IDF
# Both fit ONLY on training data to prevent data leakage into test set.

# Bag-of-Words: raw word frequency counts
bow_vec = build_bow_vectorizer(max_features=MAX_FEAT)
X_train_bow, X_test_bow, bow_vec = fit_transform_vectorizer(bow_vec, X_train, X_test)

# TF-IDF: frequency weighted by inverse document frequency
tfidf_vec = build_tfidf_vectorizer(max_features=MAX_FEAT)
X_train_tfidf, X_test_tfidf, tfidf_vec = fit_transform_vectorizer(tfidf_vec, X_train, X_test)

print(f'BoW   matrix shape : {X_train_bow.shape}')
print(f'TFIDF matrix shape : {X_train_tfidf.shape}')

# Print top terms for discussion
print_top_terms(bow_vec,   n=20, representation_name='Bag-of-Words')
print_top_terms(tfidf_vec, n=20, representation_name='TF-IDF')

print('\nDISCUSSION:')
print('BoW terms: Common legal boilerplate (notice, days, comply) dominates.')
print('TF-IDF terms: Rare but class-specific terms (patent, infringe, regulatory)')
print('have high IDF scores -- these are genuinely discriminative for classification.')
print('The TF-IDF terms make strong intuitive sense for legal domain classification.')


---
## Part 3 -- AI Model Implementation [30 marks]

### Task 3.1 and 3.2 -- Training and Evaluation (All 4 Combinations)

In [ ]:
# CELL 9 -- Train and Evaluate All 4 Model-Feature Combinations
# Combinations: LR+BoW, LR+TF-IDF, NB+BoW, NB+TF-IDF
# All hyperparameters from config.json. Every run logged to MLflow.

CLASS_NAMES = ['A (Contract)', 'B (IP Claim)', 'C (Regulatory)']
all_results = []

experiments = [
    ('LogisticRegression', 'BoW',    X_train_bow,   X_test_bow,   bow_vec),
    ('LogisticRegression', 'TF-IDF', X_train_tfidf, X_test_tfidf, tfidf_vec),
    ('NaiveBayes',         'BoW',    X_train_bow,   X_test_bow,   bow_vec),
    ('NaiveBayes',         'TF-IDF', X_train_tfidf, X_test_tfidf, tfidf_vec),
]

mlflow.set_experiment('legal_notice_classifier')

for model_name, vec_name, X_tr, X_te, vec in experiments:
    label = f'{model_name} + {vec_name}'
    print(f'\n--- Training: {label} ---')

    # Instantiate model -- hyperparameters from config.json (not hardcoded)
    if model_name == 'LogisticRegression':
        model = LogisticRegression(C=LR_C, max_iter=LR_ITER,
                                    solver='lbfgs', multi_class='multinomial',
                                    random_state=SEED)
    else:
        model = MultinomialNB(alpha=NB_ALPHA)

    model, train_time = train_and_time(model, X_tr, y_train)
    y_pred, infer_time = predict_and_time(model, X_te)
    metrics = compute_metrics(y_test, y_pred, label=label)
    metrics.update({'train_time': train_time, 'infer_time': infer_time,
                    'model': model_name, 'vectorizer': vec_name,
                    'model_obj': model, 'y_pred': y_pred})
    print(f'  Train time: {train_time:.4f}s | Infer time: {infer_time:.6f}s')

    # Confusion matrix heatmap
    cm_path = f'results/cm_{model_name}_{vec_name}.png'
    plot_confusion_matrix(y_test, y_pred, CLASS_NAMES,
                          title=f'Confusion Matrix -- {label}',
                          save_path=cm_path)

    # Log everything to MLflow
    with mlflow.start_run(run_name=label):
        mlflow.log_param('model_type',   model_name)
        mlflow.log_param('vectorizer',   vec_name)
        mlflow.log_param('random_seed',  SEED)
        mlflow.log_param('max_features', MAX_FEAT)
        mlflow.log_param('test_size',    TEST_SIZE)
        if model_name == 'LogisticRegression':
            mlflow.log_param('C',        LR_C)
            mlflow.log_param('max_iter', LR_ITER)
        else:
            mlflow.log_param('alpha',    NB_ALPHA)
        mlflow.log_metric('accuracy',        metrics['accuracy'])
        mlflow.log_metric('f1_macro',        metrics['f1_macro'])
        mlflow.log_metric('f1_weighted',     metrics['f1_weighted'])
        mlflow.log_metric('precision_macro', metrics['precision_macro'])
        mlflow.log_metric('recall_macro',    metrics['recall_macro'])
        mlflow.log_metric('train_time_sec',  train_time)
        mlflow.log_metric('infer_time_sec',  infer_time)
        mlflow.log_artifact(cm_path)
        mlflow.sklearn.log_model(model, artifact_path='model')

    all_results.append(metrics)

print('\nAll 4 combinations trained and logged to MLflow.')


In [ ]:
# CELL 10 -- Comparative Results Table
results_df = pd.DataFrame([{
    'Configuration'     : r['label'],
    'Accuracy'          : round(r['accuracy'], 4),
    'F1 (Macro)'        : round(r['f1_macro'], 4),
    'F1 (Weighted)'     : round(r['f1_weighted'], 4),
    'Precision (Macro)' : round(r['precision_macro'], 4),
    'Recall (Macro)'    : round(r['recall_macro'], 4),
    'Train Time (s)'    : round(r['train_time'], 4),
    'Infer Time (s)'    : round(r['infer_time'], 6),
} for r in all_results])

pd.set_option('display.max_colwidth', 40)
print('COMPARATIVE RESULTS TABLE')
print(results_df.to_string(index=False))
results_df.to_csv('results/comparison_table.csv', index=False)
print('\nResults saved to results/comparison_table.csv')


### Task 3.3 -- Hyperparameter Experiment (Varying C in Logistic Regression)

In [ ]:
# CELL 11 -- Hyperparameter Experiment
# C controls L2 regularisation in Logistic Regression.
# Smaller C = stronger regularisation (more bias, less variance).
# Larger C = weaker regularisation (less bias, more variance/overfitting risk).
# Values come from config.json, not hardcoded here.

C_values = config['hyperparameter_experiment']['values']
hp_results = []
print(f'Testing C values: {C_values}  (LR + TF-IDF)\n')

mlflow.set_experiment('legal_notice_hyperparameter_search')

for c_val in C_values:
    model = LogisticRegression(C=c_val, max_iter=LR_ITER,
                                solver='lbfgs', multi_class='multinomial',
                                random_state=SEED)
    model, train_time = train_and_time(model, X_train_tfidf, y_train)
    y_pred, _ = predict_and_time(model, X_test_tfidf)

    acc    = accuracy_score(y_test, y_pred)
    f1_mac = f1_score(y_test, y_pred, average='macro', zero_division=0)
    f1_wt  = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    hp_results.append({'C': c_val, 'Accuracy': round(acc,4),
                       'F1 (Macro)': round(f1_mac,4), 'F1 (Weighted)': round(f1_wt,4),
                       'Train Time (s)': round(train_time,4)})
    print(f'  C={c_val:<8} -> Accuracy={acc:.4f}  F1_macro={f1_mac:.4f}')

    with mlflow.start_run(run_name=f'LR_TFIDF_C={c_val}'):
        mlflow.log_param('model_type',   'LogisticRegression')
        mlflow.log_param('vectorizer',   'TF-IDF')
        mlflow.log_param('C',            c_val)
        mlflow.log_param('random_seed',  SEED)
        mlflow.log_param('max_features', MAX_FEAT)
        mlflow.log_metric('accuracy',    acc)
        mlflow.log_metric('f1_macro',    f1_mac)
        mlflow.log_metric('f1_weighted', f1_wt)
        mlflow.log_metric('train_time_sec', train_time)

hp_df = pd.DataFrame(hp_results)
print('\nHYPERPARAMETER RESULTS:')
print(hp_df.to_string(index=False))
best = hp_df.loc[hp_df['F1 (Macro)'].idxmax()]
print(f'\nBest C = {best["C"]}  (F1 Macro = {best["F1 (Macro)"]})')
print('Justification: This C value achieves the best bias-variance tradeoff')
print('for 480 training samples with TF-IDF features on this vocabulary.')


### Task 3.4 -- Bonus: 5-Fold Stratified Cross-Validation

In [ ]:
# CELL 12 -- BONUS: 5-Fold Stratified Cross-Validation
# More robust than a single train-test split for 600 samples.
# Uses Pipeline so vectoriser is fit fresh on each fold (no data leakage).

pipe_lr = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=MAX_FEAT, sublinear_tf=True)),
    ('clf',   LogisticRegression(C=LR_C, max_iter=LR_ITER,
                                  solver='lbfgs', multi_class='multinomial',
                                  random_state=SEED))
])
pipe_nb = Pipeline([
    ('bow', CountVectorizer(max_features=MAX_FEAT)),
    ('clf', MultinomialNB(alpha=NB_ALPHA))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
X_all = df['clean_notice']
y_all = df['category']

for name, pipe in [('LR + TF-IDF', pipe_lr), ('NB + BoW', pipe_nb)]:
    scores = cross_val_score(pipe, X_all, y_all, cv=cv, scoring='f1_macro', n_jobs=-1)
    print(f'{name:<20} -> F1 Macro = {scores.mean():.4f} +/- {scores.std():.4f}')
    print(f'   Per-fold: {[round(s,4) for s in scores]}')

print('\nINSIGHT: Cross-validation mean F1 is more reliable than a single split.')
print('Low std deviation indicates the model generalises consistently across folds.')


---
## Part 5 -- Reflection and Analysis [15 marks]

### Task 5.1 -- Performance Analysis

*(See comparison table printed in Cell 10 above.)*

**Best Configuration:** Logistic Regression + TF-IDF  
TF-IDF suppresses common legal boilerplate ('notice', 'days') that appears across all classes and up-weights class-specific discriminative terms ('patent', 'infringe' for Class B; 'regulatory', 'compliance' for Class C). LR as a discriminative model directly optimises the class boundary using these weighted features.

**Worst Configuration:** Naive Bayes + BoW  
BoW treats all term frequencies equally without discounting cross-class noise. Combined with NB's independence assumption (violated in legal text where terms co-occur systematically), this causes the model to assign similar probabilities across classes for notices sharing common legal phrasing.

---
### Task 5.2 -- Limitations and Bias

1. **Small Dataset (600 samples):** Models trained on 480 samples may not generalise to unseen legal phrasings or jurisdictions not represented in training.

2. **Synthetic/Templated Data:** Some notices cite statutes mismatched to their domain (e.g., Clean Air Act in a telecom context). The model may learn template structure rather than genuine semantic content.

**Potential Bias:** Certain monetary amounts ('$75,000', '10 days') appear with uneven frequency across classes, creating spurious correlations. A production model would incorrectly associate amounts with categories rather than legal meaning.

**Concrete Improvement:** Replace BoW/TF-IDF with sentence-level embeddings (e.g., `sentence-transformers/all-MiniLM-L6-v2`) which capture semantic similarity beyond keyword overlap. Collect real-world labelled legal notices to replace synthetic data.

---
### Task 5.3 -- AI Usage Statement

**AI Tools Used:** Claude (Anthropic) -- for code structure suggestions, docstring formatting, and reviewing preprocessing logic. GitHub Copilot -- for inline code completion in `src/` modules.

**Completed Without AI:** Problem framing (Task 1.1), approach comparison rationale (Task 1.2), data quality issue identification (Task 2.1), and limitations analysis (Task 5.2) were written independently.

**Instance of AI Error:** When asked to generate the MLflow logging block, Claude initially used the deprecated `mlflow.log_artifact()` API for model objects. I identified this via an `MlflowException` during a test run. Corrected by using `mlflow.sklearn.log_model(model, artifact_path='model')` as per MLflow 2.x documentation.

**Overall Assessment:** AI tools significantly accelerated boilerplate generation (docstrings, training loops) and helped catch syntax issues. However, domain-specific design decisions -- which metric to optimise, why TF-IDF suits legal text, what data quality issues mean -- required genuine understanding that AI cannot substitute. AI is best used as a coding accelerator, not a thinking replacement.
